# Simple example of subclassing and using the line search 



In [87]:
import jax
import jax.numpy as jnp
import optax
import flax.linen as nn
from flax.training.train_state import TrainState
from flax import struct


In [79]:
# Define our optimizer; if we comment out the linesearch, it'll not converge
solver = optax.chain(
    optax.sgd(learning_rate=1.),
    optax.scale_by_backtracking_linesearch(max_backtracking_steps=15)
)

In [88]:
type(solver)

optax._src.base.GradientTransformationExtraArgs

In [80]:
# We create a simple ax + b model to test optimizing it
model = nn.Dense(1) 
params = model.init(jax.random.key(0), jnp.array([0.0]))

{'params': {'kernel': Array([[0.08261669]], dtype=float32), 'bias': Array([0.], dtype=float32)}}


In [89]:

# The TrainState in Flax doesn't allow for more complex optimziers
# Thus we subclass it and insert the way we need to update 
class MyTrainState(TrainState):
    tx: optax.GradientTransformationExtraArgs = struct.field(pytree_node=False)
    def apply_gradients(self, *, grads, **kwargs):
        value, value_fn, x, y = kwargs['value'], kwargs['value_fn'], kwargs['x'], kwargs['y']
        updates, new_opt_state = self.tx.update(
            grads,
            self.opt_state,
            {'params': self.params},
            value=value,
            grad=grads,
            value_fn=value_fn,
            x=x,
            y=y
        )
        new_params_with_opt = optax.apply_updates(
            {'params': self.params}, updates)
        return self.replace(
          step=self.step + 1,
          params=new_params_with_opt['params'],
          opt_state=new_opt_state,
        )
        # print(new_params_with_opt)
    

In [91]:
state = MyTrainState.create(
    apply_fn=model.apply, 
    params=params['params'], # I know this is non-standard, but I like this...
    tx=solver
)

In [92]:
# Function with additional inputs other than params
def fn_state(params, x, y):
    return jnp.mean((state.apply_fn(params, x) - y) ** 2)

In [93]:
opt_state = solver.init(params)
x = jnp.linspace(0, 1, 10).reshape((-1, 1))
y = 2 * x + 4

In [96]:
for _ in range(20):    
    # Use state
    value, grad = jax.value_and_grad(fn_state)({'params': state.params}, x, y)
    state = state.apply_gradients(
        grads=grad,
        # params={'params': state.params},
        value=value,
        grad=grad,
        value_fn=fn_state,
        x=x,
        y=y
    )


In [97]:
print(state.params)

{'bias': Array([3.9951296], dtype=float32), 'kernel': Array([[2.0043876]], dtype=float32)}
